In [ ]:
# ==============================
# 0. MOUNT GOOGLE DRIVE
# ==============================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ==============================
# 1. INSTALL REQUIRED PACKAGES
# ==============================
!pip install awscli h5py scipy -q

# ==============================
# 2. IMPORTS
# ==============================
import os
import subprocess
from pathlib import Path
import shutil
import h5py
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# ==============================
# 3. PATHS
# ==============================
BASE_PATH = Path("/content/drive/MyDrive/things_fmri_project")

# persistent outputs only
PROC_PATH = BASE_PATH / "processed_trials"

# ephemeral raw storage (NEW)
RAW_TMP = Path("/content/raw_tmp")
RAW_TMP.mkdir(parents=True, exist_ok=True)


PROC_PATH.mkdir(parents=True, exist_ok=True)

# local temporary folder for large image-level arrays
LOCAL_TMP = Path("/content/image_level_tmp")
LOCAL_TMP.mkdir(parents=True, exist_ok=True)

# ==============================
# 4. SETTINGS
# ==============================
# Start with one subject first
subs = [1, 2, 3]

# Keep False at first. Turn True only after subject 1 works.
COPY_TO_DRIVE = False

# Feature chunk size for large H5 reading
CHUNK_SIZE = 2048

# ==============================
# 5. VALIDATION
# ==============================
def is_valid(sub_dir):
    stim_files = list(sub_dir.glob("*stimulus-metadata*.tsv"))
    h5_files = list(sub_dir.glob("*voxel-wise-responses*.h5"))
    return len(stim_files) > 0 and len(h5_files) > 0

# ==============================
# 6. DOWNLOAD SUBJECT DATA
# ==============================
def sync_subject(sub_str, target_dir):

    if target_dir.exists() and is_valid(target_dir):
        print(f"{sub_str} already downloaded. Skipping.")
        return

    if target_dir.exists():
        shutil.rmtree(target_dir)

    target_dir.mkdir(parents=True, exist_ok=True)

    s3_path = f"s3://openneuro.org/ds004192/derivatives/ICA-betas/{sub_str}/voxel-metadata/"

    print(f"\nSyncing {sub_str}...")

    result = subprocess.run([
        "aws", "s3", "sync",
        s3_path,
        str(target_dir),
        "--no-sign-request"
    ])

    if result.returncode != 0:
        raise RuntimeError(f"S3 sync failed for {sub_str}")

    print(f"Download complete: {sub_str}")

# ==============================
# 7. RUN DOWNLOAD
# ==============================
for sub in subs:
    sub_str = f"sub-0{sub}"
    target_dir = RAW_TMP / sub_str / "voxel-metadata"
    sync_subject(sub_str, target_dir)

# ==============================
# 8. SAFE METADATA LOADER
# ==============================
def load_metadata(path):
    # try TSV first
    df = pd.read_csv(path, sep="\t")

    # fallback if needed
    if len(df.columns) == 1:
        df = pd.read_csv(path, sep=",")

    print("Metadata columns:", list(df.columns))
    return df

# ==============================
# 9. BUILD TRUE IMAGE-LEVEL DATA
# ==============================
saved_output_paths = {}

for sub in subs:

    sub_str = f"sub-0{sub}"
    save_dir = PROC_PATH / sub_str
    save_dir.mkdir(parents=True, exist_ok=True)

    print("\n==============================")
    print(f"PROCESSING {sub_str}")
    print("==============================")

    raw_dir = RAW_TMP / sub_str / "voxel-metadata"

    stim_path = list(raw_dir.glob("*stimulus-metadata*.tsv"))[0]
    h5_path = list(raw_dir.glob("*voxel-wise-responses*.h5"))[0]

    # ---- LOAD METADATA ----
    stim_meta = load_metadata(stim_path)

    required_cols = ["trial_id", "stimulus", "concept"]
    for c in required_cols:
        if c not in stim_meta.columns:
            raise RuntimeError(f"Missing column: {c}")

    stim_meta = stim_meta.reset_index(drop=True)
    n_trials = len(stim_meta)

    # Save full metadata
    stim_meta.to_csv(save_dir / "stimulus_metadata.csv", index=False)

    # Trial index table
    trial_index = stim_meta[["trial_id", "stimulus", "concept"]].copy()
    trial_index["trial_col"] = np.arange(n_trials)
    trial_index.to_csv(save_dir / "trial_index.csv", index=False)

    # ---- BUILD IMAGE GROUPING ----
    # Keep stimulus order exactly as it appears in metadata
    image_ids, unique_stimuli = pd.factorize(trial_index["stimulus"], sort=False)
    n_images = len(unique_stimuli)

    # safety check: one exact stimulus should map to one concept
    concept_check = trial_index.groupby("stimulus", sort=False)["concept"].nunique()
    if (concept_check > 1).any():
        bad_stimulus = concept_check[concept_check > 1].index[0]
        raise RuntimeError(f"Stimulus {bad_stimulus} maps to multiple concepts.")

    image_counts = np.bincount(image_ids, minlength=n_images).astype(np.int32)

    image_index = (
        trial_index.groupby("stimulus", sort=False)
        .agg(
            concept=("concept", "first"),
            n_presentations=("stimulus", "size")
        )
        .reset_index()
    )
    image_index["image_row"] = np.arange(n_images)
    image_index.to_csv(save_dir / "image_index.csv", index=False)

    print("Metadata rows (trials):", n_trials)
    print("Unique stimuli:", n_images)

    # Sparse averaging matrix:
    # each trial contributes 1 / number_of_presentations to its image
    weights = (1.0 / image_counts[image_ids]).astype(np.float32)
    A = csr_matrix(
        (weights, (np.arange(n_trials), image_ids)),
        shape=(n_trials, n_images),
        dtype=np.float32
    )

    # ---- OPEN H5 AND DETECT ORIENTATION ----
    with h5py.File(h5_path, "r") as f:
        Y_raw = f["ResponseData"]["block0_values"]
        raw_shape = Y_raw.shape

        print("Raw H5 shape:", raw_shape)

        if raw_shape[1] == n_trials:
            orientation = "features_by_trials"
            n_features = raw_shape[0]
        elif raw_shape[0] == n_trials:
            orientation = "trials_by_features"
            n_features = raw_shape[1]
        else:
            raise RuntimeError(
                f"Could not match metadata rows ({n_trials}) to H5 shape {raw_shape} for {sub_str}."
            )

        print("Detected orientation:", orientation)
        print("Feature dimension:", n_features)

        # ---- SAVE LARGE OUTPUT LOCALLY FIRST ----
        out_path_local = LOCAL_TMP / f"Y_{sub_str}_images.npy"

        Y_image = np.lib.format.open_memmap(
            out_path_local,
            mode="w+",
            dtype=np.float32,
            shape=(n_images, n_features)
        )

        n_chunks = (n_features + CHUNK_SIZE - 1) // CHUNK_SIZE

        if orientation == "features_by_trials":
            # Y_raw shape = [features, trials]
            for chunk_idx, start in enumerate(range(0, n_features, CHUNK_SIZE)):
                end = min(start + CHUNK_SIZE, n_features)

                # block shape: [feature_chunk, trials]
                block = np.asarray(Y_raw[start:end, :], dtype=np.float32)

                # average repeated identical-image columns
                # A.T shape = [n_images, trials]
                # block.T shape = [trials, feature_chunk]
                # result = [n_images, feature_chunk]
                block_img = A.T @ block.T

                Y_image[:, start:end] = np.asarray(block_img, dtype=np.float32)

                if chunk_idx % 20 == 0 or end == n_features:
                    print(f"Processed chunk {chunk_idx + 1}/{n_chunks} | rows {start}:{end}")

        else:
            # Y_raw shape = [trials, features]
            for chunk_idx, start in enumerate(range(0, n_features, CHUNK_SIZE)):
                end = min(start + CHUNK_SIZE, n_features)

                # block shape: [trials, feature_chunk]
                block = np.asarray(Y_raw[:, start:end], dtype=np.float32)

                # A.T @ block -> [n_images, feature_chunk]
                block_img = A.T @ block

                Y_image[:, start:end] = np.asarray(block_img, dtype=np.float32)

                if chunk_idx % 20 == 0 or end == n_features:
                    print(f"Processed chunk {chunk_idx + 1}/{n_chunks} | cols {start}:{end}")

        # flush local file
        del Y_image

    # ---- OPTIONAL COPY TO DRIVE ----
    if COPY_TO_DRIVE:
        out_path_drive = save_dir / f"Y_{sub_str}_images.npy"
        shutil.copy2(out_path_local, out_path_drive)
        saved_output_paths[sub_str] = out_path_drive
        print(f"Copied image-level dataset to Drive: {out_path_drive}")
    else:
        saved_output_paths[sub_str] = out_path_local
        print(f"Saved image-level dataset locally: {out_path_local}")

# optional but recommended cleanup
shutil.rmtree(RAW_TMP, ignore_errors=True)
RAW_TMP.mkdir(parents=True, exist_ok=True)

# ==============================
# 10. FINAL CHECK
# ==============================
print("\n===== FINAL CHECK =====")

for sub in subs:
    sub_str = f"sub-0{sub}"
    save_dir = PROC_PATH / sub_str

    image_meta = pd.read_csv(save_dir / "image_index.csv")
    y_path = saved_output_paths[sub_str]
    Y_image = np.load(y_path, mmap_mode="r")

    print(f"\n{sub_str}")
    print("Image-level Y path:", y_path)
    print("Image-level Y shape:", Y_image.shape)
    print("Image metadata shape:", image_meta.shape)
    print("Stimulus examples:", image_meta["stimulus"].head().tolist())
    print("Concept examples:", image_meta["concept"].head().tolist())
    print("Presentation counts:", image_meta["n_presentations"].head().tolist())

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.

Syncing sub-01...
Download complete: sub-01

Syncing sub-02...
Download complete: sub-02

Syncing sub-03...
Download complete: sub-03

PROCESSING sub-01
Metadata columns: ['trial_type', 'session', 'run', 'subject_id', 'trial_id', 'stimulus', 'concept']
Metadata rows (trials): 9840
Unique stimuli: 8740
Raw H5 shape: (211339, 9840)
Detected orientation: features_by_trials
Feature dimensio

In [ ]:
# ==============================
# 11. IMAGE REPETITION SUMMARY
# ==============================
for sub in subs:
    sub_str = f"sub-0{sub}"
    save_dir = PROC_PATH / sub_str
    image_meta = pd.read_csv(save_dir / "image_index.csv")

    print(f"\n{sub_str}")
    print("Unique stimuli:", image_meta["stimulus"].nunique())
    print("Top repetition counts:")
    print(image_meta["n_presentations"].value_counts().sort_index().tail())


sub-01
Unique stimuli: 8740
Top repetition counts:
n_presentations
1     8640
12     100
Name: count, dtype: int64

sub-02
Unique stimuli: 8740
Top repetition counts:
n_presentations
1     8640
12     100
Name: count, dtype: int64

sub-03
Unique stimuli: 8740
Top repetition counts:
n_presentations
1     8640
12     100
Name: count, dtype: int64


In [ ]:
# ==============================
# CHECK 1B: SAME IMAGES, DIFFERENT ORDER?
# ==============================
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/things_fmri_project/processed_trials")

idx1 = pd.read_csv(BASE / "sub-01" / "image_index.csv")
idx2 = pd.read_csv(BASE / "sub-02" / "image_index.csv")
idx3 = pd.read_csv(BASE / "sub-03" / "image_index.csv")


s1 = set(idx1["stimulus"])
s2 = set(idx2["stimulus"])
s3 = set(idx3["stimulus"])

print("sub-01 vs sub-02 same stimulus set:", s1 == s2)
print("sub-01 vs sub-03 same stimulus set:", s1 == s3)
print("sub-02 vs sub-03 same stimulus set:", s2 == s3)

print("sub-01 vs sub-02 same concept set :", set(idx1["concept"]) == set(idx2["concept"]))
print("sub-01 vs sub-03 same concept set :", set(idx1["concept"]) == set(idx3["concept"]))
print("sub-02 vs sub-03 same concept set :", set(idx2["concept"]) == set(idx3["concept"]))

sub-01 vs sub-02 same stimulus set: True
sub-01 vs sub-03 same stimulus set: True
sub-02 vs sub-03 same stimulus set: True
sub-01 vs sub-02 same concept set : True
sub-01 vs sub-03 same concept set : True
sub-02 vs sub-03 same concept set : True


In [ ]:
# ==============================
# CHECK 2: IMAGES PER CONCEPT
# ==============================
concept_counts = idx1.groupby("concept")["stimulus"].nunique().sort_values()

print("Total concepts:", concept_counts.shape[0])
print("\nSmallest image counts per concept:")
print(concept_counts.head(10))

print("\nLargest image counts per concept:")
print(concept_counts.tail(10))

print("\nConcept count summary:")
print(concept_counts.describe())

Total concepts: 720

Smallest image counts per concept:
concept
plastic_film       12
pie                12
pig                12
pigeon             12
pill               12
pillow             12
pineapple          12
ping-pong_table    12
pinwheel           12
pistachio          12
Name: stimulus, dtype: int64

Largest image counts per concept:
concept
streetlight    13
butterfly      13
nest           13
starfish       13
stalagmite     13
candelabra     13
fudge          13
spoon          13
speaker        13
footprint      13
Name: stimulus, dtype: int64

Concept count summary:
count    720.000000
mean      12.138889
std        0.346071
min       12.000000
25%       12.000000
50%       12.000000
75%       12.000000
max       13.000000
Name: stimulus, dtype: float64


In [ ]:
# ==============================
# CHECK 3: REPEATED-IMAGE CONSISTENCY (FIXED)
# ==============================
import h5py
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

BASE = Path("/content/drive/MyDrive/things_fmri_project")
RAW = BASE / "raw" / "sub-01" / "voxel-metadata"
PROC = BASE / "processed_trials" / "sub-01"

trial_index = pd.read_csv(PROC / "trial_index.csv")
h5_path = list(RAW.glob("*voxel-wise-responses*.h5"))[0]

rep = trial_index["stimulus"].value_counts()
rep_stimuli = rep[rep > 1].index.tolist()

print("Number of repeated exact images:", len(rep_stimuli))

np.random.seed(42)

with h5py.File(h5_path, "r") as f:
    Y_raw = f["ResponseData"]["block0_values"]
    n_features = Y_raw.shape[0]

    feat_idx = np.sort(np.random.choice(n_features, size=3000, replace=False))

    sims = []

    for stim in rep_stimuli[:100]:
        cols = np.sort(trial_index.loc[trial_index["stimulus"] == stim, "trial_col"].values)

        block = np.asarray(Y_raw[feat_idx, :][:, cols], dtype=np.float32).T
        S = cosine_similarity(block)

        mask = ~np.eye(S.shape[0], dtype=bool)
        sims.append(S[mask].mean())

print("Mean repeated-image similarity:", float(np.mean(sims)))
print("Std repeated-image similarity :", float(np.std(sims)))

Number of repeated exact images: 100
Mean repeated-image similarity: 0.022006100043654442
Std repeated-image similarity : 0.01155759859830141


In [ ]:
# ==============================
# CHECK 4: FAST WITHIN vs BETWEEN CONCEPT
# ==============================
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

BASE = Path("/content/drive/MyDrive/things_fmri_project/processed_trials/sub-01")

Y = np.load("/content/image_level_tmp/Y_sub-01_images.npy", mmap_mode="r")
meta = pd.read_csv(BASE / "image_index.csv")

np.random.seed(42)

# sample images for speed
img_idx = np.random.choice(Y.shape[0], size=3000, replace=False)

# sample features for speed
feat_idx = np.random.choice(Y.shape[1], size=1500, replace=False)

Y_small = np.asarray(Y[img_idx][:, feat_idx], dtype=np.float32)
labels = meta.iloc[img_idx]["concept"].values

concepts = np.unique(labels)
prototypes = {}

for c in concepts:
    idx = np.where(labels == c)[0]
    if len(idx) > 0:
        prototypes[c] = Y_small[idx].mean(axis=0, keepdims=True)

within = []
between = []

for c in concepts:
    idx = np.where(labels == c)[0]
    if len(idx) < 2:
        continue

    sims_w = cosine_similarity(Y_small[idx], prototypes[c]).mean()
    within.append(sims_w)

    other_choices = concepts[concepts != c]
    other = np.random.choice(other_choices)
    sims_b = cosine_similarity(Y_small[idx], prototypes[other]).mean()
    between.append(sims_b)

print("Mean within-concept similarity :", float(np.mean(within)))
print("Mean between-concept similarity:", float(np.mean(between)))
print("Gap (within - between)         :", float(np.mean(within) - np.mean(between)))

Mean within-concept similarity : 0.5230473875999451
Mean between-concept similarity: 0.04276513308286667
Gap (within - between)         : 0.4802822470664978


In [ ]:
# ==========================================
# CHECK A: LEAVE-ONE-OUT WITHIN vs BETWEEN
# small random sample of concepts
# ==========================================
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

# ----- SETTINGS -----
sub = "sub-01"
n_concepts_sample = 80     # small random sample of concepts
n_features_sample = 1500   # feature subset for speed
np.random.seed(42)

# ----- PATHS -----
BASE = Path("/content/drive/MyDrive/things_fmri_project/processed_trials") / sub
meta = pd.read_csv(BASE / "image_index.csv")
Y = np.load(f"/content/image_level_tmp/Y_{sub}_images.npy", mmap_mode="r")

# ----- SAMPLE CONCEPTS -----
all_concepts = meta["concept"].unique()
sampled_concepts = np.random.choice(
    all_concepts, size=min(n_concepts_sample, len(all_concepts)), replace=False
)

mask = meta["concept"].isin(sampled_concepts).values
meta_sub = meta[mask].reset_index(drop=True)

# sample features for speed
feat_idx = np.sort(np.random.choice(Y.shape[1], size=n_features_sample, replace=False))
Y_sub = np.asarray(Y[mask][:, feat_idx], dtype=np.float32)

labels = meta_sub["concept"].values
concepts = np.unique(labels)

within_scores = []
between_scores = []

for c in concepts:
    idx_c = np.where(labels == c)[0]
    if len(idx_c) < 2:
        continue

    other_concepts = concepts[concepts != c]
    other_c = np.random.choice(other_concepts)
    idx_other = np.where(labels == other_c)[0]

    for i in idx_c:
        # leave-one-out prototype for same concept
        idx_loo = idx_c[idx_c != i]
        if len(idx_loo) == 0:
            continue

        proto_within = Y_sub[idx_loo].mean(axis=0, keepdims=True)
        sim_within = cosine_similarity(Y_sub[i:i+1], proto_within)[0, 0]
        within_scores.append(sim_within)

        # random other concept prototype
        proto_between = Y_sub[idx_other].mean(axis=0, keepdims=True)
        sim_between = cosine_similarity(Y_sub[i:i+1], proto_between)[0, 0]
        between_scores.append(sim_between)

print("Sampled concepts:", len(concepts))
print("Mean leave-one-out within-concept similarity :", float(np.mean(within_scores)))
print("Mean between-concept similarity              :", float(np.mean(between_scores)))
print("Gap (within - between)                       :", float(np.mean(within_scores) - np.mean(between_scores)))

Sampled concepts: 80
Mean leave-one-out within-concept similarity : 0.06889760494232178
Mean between-concept similarity              : 0.06653989106416702
Gap (within - between)                       : 0.0023577138781547546


In [ ]:
# ==========================================
# CHECK B1: REPEATED-IMAGE SIMILARITY
# few repeated exact images within one subject
# ==========================================
import h5py
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

# ----- SETTINGS -----
sub = "sub-01"
n_repeated_images = 10
n_features_sample = 3000
np.random.seed(42)

# ----- PATHS -----
BASE = Path("/content/drive/MyDrive/things_fmri_project")
RAW = BASE / "raw" / sub / "voxel-metadata"
PROC = BASE / "processed_trials" / sub

trial_index = pd.read_csv(PROC / "trial_index.csv")
h5_path = list(RAW.glob("*voxel-wise-responses*.h5"))[0]

# repeated exact images only
rep = trial_index["stimulus"].value_counts()
rep_stimuli = rep[rep > 1].index.tolist()

sampled_rep = np.random.choice(
    rep_stimuli, size=min(n_repeated_images, len(rep_stimuli)), replace=False
)

with h5py.File(h5_path, "r") as f:
    Y_raw = f["ResponseData"]["block0_values"]

    # based on your current result, orientation is features_by_trials
    n_features = Y_raw.shape[0]
    feat_idx = np.sort(np.random.choice(n_features, size=n_features_sample, replace=False))

    print("Sampled repeated images:\n")

    for stim in sampled_rep:
        cols = np.sort(trial_index.loc[trial_index["stimulus"] == stim, "trial_col"].values)

        # block: repeated_trials x sampled_features
        block = np.asarray(Y_raw[feat_idx, :][:, cols], dtype=np.float32).T

        S = cosine_similarity(block)
        mask = ~np.eye(S.shape[0], dtype=bool)
        mean_sim = S[mask].mean()

        print(f"{stim:25s} | n_presentations = {len(cols):2d} | mean similarity = {mean_sim:.4f}")

Sampled repeated images:

hippopotamus_16s.jpg      | n_presentations = 12 | mean similarity = 0.0279
shredder_13s.jpg          | n_presentations = 12 | mean similarity = 0.0152
candelabra_14s.jpg        | n_presentations = 12 | mean similarity = 0.0170
chest1_14s.jpg            | n_presentations = 12 | mean similarity = 0.0289
microscope_14n.jpg        | n_presentations = 12 | mean similarity = 0.0304
clipboard_14s.jpg         | n_presentations = 12 | mean similarity = 0.0195
ferris_wheel_20s.jpg      | n_presentations = 12 | mean similarity = 0.0276
monkey_18n.jpg            | n_presentations = 12 | mean similarity = 0.0283
bean_13s.jpg              | n_presentations = 12 | mean similarity = 0.0155
dragonfly_13s.jpg         | n_presentations = 12 | mean similarity = 0.0172


In [ ]:
# ==========================================
# CHECK B2: REPEATED-IMAGE SIMILARITY
# compare across subjects
# ==========================================
import h5py
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

BASE = Path("/content/drive/MyDrive/things_fmri_project")
subs = ["sub-01", "sub-02", "sub-03"]

n_features_sample = 3000
np.random.seed(42)

for sub in subs:
    RAW = BASE / "raw" / sub / "voxel-metadata"
    PROC = BASE / "processed_trials" / sub

    trial_index = pd.read_csv(PROC / "trial_index.csv")
    h5_path = list(RAW.glob("*voxel-wise-responses*.h5"))[0]

    rep = trial_index["stimulus"].value_counts()
    rep_stimuli = rep[rep > 1].index.tolist()

    with h5py.File(h5_path, "r") as f:
        Y_raw = f["ResponseData"]["block0_values"]
        n_features = Y_raw.shape[0]

        feat_idx = np.sort(np.random.choice(n_features, size=min(n_features_sample, n_features), replace=False))

        sims = []
        for stim in rep_stimuli:
            cols = np.sort(trial_index.loc[trial_index["stimulus"] == stim, "trial_col"].values)
            block = np.asarray(Y_raw[feat_idx, :][:, cols], dtype=np.float32).T

            S = cosine_similarity(block)
            mask = ~np.eye(S.shape[0], dtype=bool)
            sims.append(S[mask].mean())

    print(f"{sub} -> mean repeated-image similarity: {float(np.mean(sims)):.4f} | std: {float(np.std(sims)):.4f}")

sub-01 -> mean repeated-image similarity: 0.0220 | std: 0.0116
sub-02 -> mean repeated-image similarity: 0.0227 | std: 0.0113
sub-03 -> mean repeated-image similarity: 0.0123 | std: 0.0082


In [ ]:
# ==============================
# 0. MOUNT DRIVE
# ==============================
from google.colab import drive
drive.mount('/content/drive')

# ==============================
# 1. IMPORTS
# ==============================
import os
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

# ==============================
# 2. PATHS (EDIT THESE)
# ==============================

# metadata (already aligned with your Y)
META_PATH = Path("/content/drive/MyDrive/things_fmri_project/processed_trials/sub-01/stimulus_metadata.csv")

# image roots (UPLOAD THESE TO DRIVE FIRST)
IMG_ROOT_1 = Path("/content/drive/MyDrive/things_1half")
IMG_ROOT_2 = Path("/content/drive/MyDrive/things_2half")

SAVE_PATH = Path("/content/drive/MyDrive/things_fmri_project/embeddings")
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# ==============================
# 3. LOAD METADATA
# ==============================
meta = pd.read_csv(META_PATH)

stimuli = meta["stimulus"].tolist()

print("Total trials:", len(stimuli))

# ==============================
# 4. LOAD CLIP
# ==============================
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

model.eval()

# ==============================
# 5. HELPER: FIND IMAGE PATH
# ==============================
def find_image(stimulus):
    concept = stimulus.split("_")[0]

    p1 = IMG_ROOT_1 / concept / stimulus
    if p1.exists():
        return p1

    p2 = IMG_ROOT_2 / concept / stimulus
    if p2.exists():
        return p2

    return None

# ==============================
# 6. COMPUTE EMBEDDINGS
# ==============================
embeddings = []
missing = []

for stim in tqdm(stimuli):

    img_path = find_image(stim)

    if img_path is None:
        missing.append(stim)
        embeddings.append(np.zeros(512))
        continue

    image = Image.open(img_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.get_image_features(**inputs)

    # handle weird return types
    if hasattr(outputs, "pooler_output"):
        emb = outputs.pooler_output
    else:
        emb = outputs

    # normalize
    emb = emb / emb.norm(dim=-1, keepdim=True)

    emb = emb.cpu().numpy().squeeze()

    embeddings.append(emb)

# ==============================
# 7. FINAL MATRIX
# ==============================
X = np.vstack(embeddings)

print("Embedding matrix shape:", X.shape)

# ==============================
# 8. SAVE
# ==============================
np.save(SAVE_PATH / "X_clip_embeddings.npy", X)

pd.Series(missing).to_csv(SAVE_PATH / "missing_images.csv", index=False)

print("Missing images:", len(missing))
print("Saved embeddings.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total trials: 9840


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 9840/9840 [2:37:11<00:00,  1.04it/s]


Embedding matrix shape: (9840, 512)
Missing images: 792
Saved embeddings.


In [ ]:
import numpy as np
import pandas as pd

# ==============================
# LOAD SMALL OBJECTS FIRST
# ==============================
X = np.load("/content/drive/MyDrive/things_fmri_project/embeddings/X_clip_embeddings.npy")
meta = pd.read_csv("/content/drive/MyDrive/things_fmri_project/processed_trials/sub-01/stimulus_metadata.csv")
image_index = pd.read_csv("/content/drive/MyDrive/things_fmri_project/processed_trials/sub-01/image_index.csv")

print("Loaded X:", X.shape)

# ==============================
# STEP 1: BUILD STIMULUS ENCODING
# ==============================
stimulus = meta["stimulus"].values

unique_stim, stim_ids = np.unique(stimulus, return_inverse=True)
n_images = len(unique_stim)
dim = X.shape[1]

print("Unique images:", n_images)

# map stimulus → row index in image space
stim_to_img = {s: i for i, s in enumerate(unique_stim)}

# ==============================
# STEP 2: CHUNKED ACCUMULATION
# ==============================
chunk_size = 2048

X_sum = np.zeros((n_images, dim), dtype=np.float32)
counts = np.zeros(n_images, dtype=np.int32)

n_trials = X.shape[0]

for start in range(0, n_trials, chunk_size):

    end = min(start + chunk_size, n_trials)

    X_chunk = X[start:end]              # (chunk, 512)
    stim_chunk = stim_ids[start:end]   # (chunk,)

    # accumulate per chunk
    for i in range(end - start):
        idx = stim_chunk[i]
        X_sum[idx] += X_chunk[i]
        counts[idx] += 1

    print(f"Processed {start}:{end}")

# ==============================
# STEP 3: FINAL AVERAGE
# ==============================
counts[counts == 0] = 1
X_image = X_sum / counts[:, None]

print("Aggregated CLIP shape:", X_image.shape)

# ==============================
# STEP 4: ALIGN TO IMAGE INDEX ORDER
# ==============================
order = np.array([stim_to_img[s] for s in image_index["stimulus"]])

X_image = X_image[order]

# ==============================
# STEP 5: FINAL CLEANUP MASK
# ==============================
valid = np.linalg.norm(X_image, axis=1) > 0

X_final = X_image[valid]

print("Final shape:", X_final.shape)

# ==============================
# SAVE SAFELY
# ==============================
np.save("/content/X_clip_aligned.npy", X_final)

print("Saved X_clip_aligned.npy")

Loaded X: (9840, 512)
Unique images: 8740
Processed 0:2048
Processed 2048:4096
Processed 4096:6144
Processed 6144:8192
Processed 8192:9840
Aggregated CLIP shape: (8740, 512)
Final shape: (8003, 512)
Saved X_clip_aligned.npy


In [ ]:
print("Original image_index:", 8740)
print("Final kept:", 8003)
print("Dropped:", 8740 - 8003)

# check how many zeros actually exist
zero_count = np.sum(np.linalg.norm(X_image, axis=1) == 0)
print("Zero embeddings detected:", zero_count)

Original image_index: 8740
Final kept: 8003
Dropped: 737
Zero embeddings detected: 737


In [ ]:
import h5py

h5_path = "/content/drive/MyDrive/things_fmri_project/raw/sub-01/voxel-metadata/sub-01_task-things_voxel-wise-responses.h5"

with h5py.File(h5_path, "r") as f:
    print("TOP KEYS:", list(f.keys()))

    def explore(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"DATASET: {name} -> shape {obj.shape}")
        else:
            print(f"GROUP: {name}")

    f.visititems(explore)

TOP KEYS: ['ResponseData']
GROUP: ResponseData
DATASET: ResponseData/axis0 -> shape (1,)
DATASET: ResponseData/axis1 -> shape (211339,)
DATASET: ResponseData/block0_items -> shape (1,)
DATASET: ResponseData/block0_values -> shape (211339, 9840)
DATASET: ResponseData/block1_items -> shape (1,)
DATASET: ResponseData/block1_values -> shape (211339, 1)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/things_fmri_project")
PROC = BASE / "processed_trials"

subs = ["sub-01", "sub-02", "sub-03"]

# ----------------------------
# 1. load original image indices
# ----------------------------
img_idx = pd.read_csv(PROC / "sub-01" / "image_index.csv")

# IMPORTANT: stimulus is shared across subjects (we verified this)
stimuli_full = img_idx["stimulus"].values

# ----------------------------
# 2. load final X stimuli mask
# ----------------------------
# You must have saved this when building X
# (if not, we reconstruct via ordering assumption)

X = np.load("/content/X_clip_aligned.npy")

# final kept indices should have been saved earlier;
# if not, we reconstruct by loading missing list

missing_df = pd.read_csv("/content/drive/MyDrive/things_fmri_project/embeddings/missing_images.csv")

missing_set = set(missing_df.iloc[:, 0].values)

# mask of valid stimuli
valid_mask = np.array([s not in missing_set for s in stimuli_full])

img_idx_aligned = img_idx[valid_mask].reset_index(drop=True)

print("Original:", len(img_idx))
print("Aligned :", len(img_idx_aligned))

Original: 8740
Aligned : 8003


In [ ]:
# rebuild stimulus order for X
final_stimuli = img_idx_aligned["stimulus"].values

# sanity check
assert len(final_stimuli) == X.shape[0]

In [ ]:
def align_subject_Y(sub):
    base = PROC / sub

    Y_path = base / f"Y_{sub}_images.npy"
    img_idx_path = base / "image_index.csv"

    Y = np.load(Y_path, mmap_mode="r")
    idx = pd.read_csv(img_idx_path)

    # map stimulus → row index
    stim_to_row = {s: i for i, s in enumerate(idx["stimulus"])}

    # build index into Y
    rows = [stim_to_row[s] for s in final_stimuli if s in stim_to_row]

    Y_aligned = Y[rows]

    print(sub, ":", Y.shape, "→", Y_aligned.shape)

    np.save(base / f"Y_{sub}_aligned.npy", Y_aligned)

    return Y_aligned

In [ ]:
Y1 = align_subject_Y("sub-01")
Y2 = align_subject_Y("sub-02")
Y3 = align_subject_Y("sub-03")

sub-01 : (8740, 211339) → (8003, 211339)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/things_fmri_project/processed_trials/sub-02/Y_sub-02_images.npy'

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/things_fmri_project")
TMP = Path("/content/image_level_tmp")
PROC = BASE / "processed_trials"

def align_subject_Y_index(sub):

    base = PROC / sub

    # ---- rebuild final_stimuli (from sub-01 reference) ----
    img_idx = pd.read_csv(PROC / "sub-01" / "image_index.csv")
    missing_df = pd.read_csv(BASE / "embeddings" / "missing_images.csv")

    missing_set = set(missing_df.iloc[:, 0].values)
    valid_mask = ~img_idx["stimulus"].isin(missing_set)

    final_stimuli = img_idx.loc[valid_mask, "stimulus"].values

    print(f"{sub}: using {len(final_stimuli)} aligned stimuli")

    # ---- locate Y ----
    drive_path = base / f"Y_{sub}_images.npy"
    tmp_path = TMP / f"Y_{sub}_images.npy"

    if drive_path.exists():
        Y_path = drive_path
        print(sub, "using DRIVE file")
    elif tmp_path.exists():
        Y_path = tmp_path
        print(sub, "using TMP file")
    else:
        raise FileNotFoundError(f"No Y file found for {sub}")

    # ---- load index ----
    idx = pd.read_csv(base / "image_index.csv")

    stim_to_row = {s: i for i, s in enumerate(idx["stimulus"])}

    missing = [s for s in final_stimuli if s not in stim_to_row]
    if len(missing) > 0:
        raise ValueError(f"{sub}: Missing stimuli in Y index: {len(missing)}")

    rows = np.array([stim_to_row[s] for s in final_stimuli], dtype=np.int32)

    print(f"{sub}: index mapping ready → shape {rows.shape}")

    # ---- SAVE ONLY INDICES (tiny file) ----
    idx_save_path = base / f"Y_{sub}_rows.npy"
    np.save(idx_save_path, rows)

    print(f"{sub}: saved row indices → {idx_save_path}")

    # ---- return lightweight handle ----
    return {
        "Y_path": str(Y_path),
        "rows_path": str(idx_save_path),
        "n_rows": len(rows)
    }

In [ ]:
info1 = align_subject_Y_index("sub-01")
info2 = align_subject_Y_index("sub-02")
info3 = align_subject_Y_index("sub-03")

sub-01: using 8003 aligned stimuli
sub-01 using DRIVE file
sub-01: index mapping ready → shape (8003,)
sub-01: saved row indices → /content/drive/MyDrive/things_fmri_project/processed_trials/sub-01/Y_sub-01_rows.npy
sub-02: using 8003 aligned stimuli
sub-02 using TMP file
sub-02: index mapping ready → shape (8003,)
sub-02: saved row indices → /content/drive/MyDrive/things_fmri_project/processed_trials/sub-02/Y_sub-02_rows.npy
sub-03: using 8003 aligned stimuli
sub-03 using TMP file
sub-03: index mapping ready → shape (8003,)
sub-03: saved row indices → /content/drive/MyDrive/things_fmri_project/processed_trials/sub-03/Y_sub-03_rows.npy


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/things_fmri_project")
PROC = BASE / "processed_trials"
TMP = Path("/content/image_level_tmp")

# ---- rebuild final_stimuli (same logic as before) ----
img_idx = pd.read_csv(PROC / "sub-01" / "image_index.csv")
missing_df = pd.read_csv(BASE / "embeddings" / "missing_images.csv")

missing_set = set(missing_df.iloc[:, 0].values)
valid_mask = ~img_idx["stimulus"].isin(missing_set)

final_stimuli = img_idx.loc[valid_mask, "stimulus"].values

print("Final stimuli count:", len(final_stimuli))

# ---- check X alignment ----
X = np.load("/content/X_clip_aligned.npy")

print("\nX shape:", X.shape)

print("\nFirst 10 stimuli (reference):")
print(final_stimuli[:10])

# ---- check each subject ----
for sub in ["sub-01", "sub-02", "sub-03"]:

    print(f"\n===== {sub} =====")

    rows = np.load(PROC / sub / f"Y_{sub}_rows.npy")

    # load Y memmap
    y_drive = PROC / sub / f"Y_{sub}_images.npy"
    y_tmp = TMP / f"Y_{sub}_images.npy"

    if y_drive.exists():
        Y = np.load(y_drive, mmap_mode="r")
    else:
        Y = np.load(y_tmp, mmap_mode="r")

    # load that subject's index
    idx = pd.read_csv(PROC / sub / "image_index.csv")

    # map rows back to stimuli
    aligned_stimuli = idx.iloc[rows]["stimulus"].values

    print("First 10 aligned stimuli:")
    print(aligned_stimuli[:10])

    # strict equality check
    match = np.all(aligned_stimuli == final_stimuli)

    print("Alignment match:", match)

Final stimuli count: 8003

X shape: (8003, 512)

First 10 stimuli (reference):
['dog_12s.jpg' 'mango_12s.jpg' 'spatula_12s.jpg' 'candelabra_14s.jpg'
 'panda_12s.jpg' 'pacifier_14s.jpg' 'crayfish_12s.jpg' 'shirt_12s.jpg'
 'mango_13s.jpg' 'cow_12s.jpg']

===== sub-01 =====
First 10 aligned stimuli:
['dog_12s.jpg' 'mango_12s.jpg' 'spatula_12s.jpg' 'candelabra_14s.jpg'
 'panda_12s.jpg' 'pacifier_14s.jpg' 'crayfish_12s.jpg' 'shirt_12s.jpg'
 'mango_13s.jpg' 'cow_12s.jpg']
Alignment match: True

===== sub-02 =====
First 10 aligned stimuli:
['dog_12s.jpg' 'mango_12s.jpg' 'spatula_12s.jpg' 'candelabra_14s.jpg'
 'panda_12s.jpg' 'pacifier_14s.jpg' 'crayfish_12s.jpg' 'shirt_12s.jpg'
 'mango_13s.jpg' 'cow_12s.jpg']
Alignment match: True

===== sub-03 =====
First 10 aligned stimuli:
['dog_12s.jpg' 'mango_12s.jpg' 'spatula_12s.jpg' 'candelabra_14s.jpg'
 'panda_12s.jpg' 'pacifier_14s.jpg' 'crayfish_12s.jpg' 'shirt_12s.jpg'
 'mango_13s.jpg' 'cow_12s.jpg']
Alignment match: True


In [ ]:
import numpy as np

np.random.seed(42)

N = 8003
indices = np.arange(N)

np.random.shuffle(indices)

split = int(0.8 * N)

train_idx = indices[:split]
test_idx = indices[split:]

print("Train:", len(train_idx), "Test:", len(test_idx))

Train: 6402 Test: 1601


In [ ]:
from sklearn.preprocessing import StandardScaler

X = np.load("/content/X_clip_aligned.npy")

scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X[train_idx])
X_test = scaler_X.transform(X[test_idx])

In [ ]:
def compute_Y_stats(Y, rows, train_idx, chunk_size=2048):

    n_features = Y.shape[1]

    mean = np.zeros(n_features, dtype=np.float32)
    var = np.zeros(n_features, dtype=np.float32)

    n_total = 0

    for start in range(0, len(train_idx), chunk_size):
        end = min(start + chunk_size, len(train_idx))

        idx = train_idx[start:end]
        y_chunk = np.asarray(Y[rows[idx]], dtype=np.float32)

        n = y_chunk.shape[0]

        mean_chunk = y_chunk.mean(axis=0)
        var_chunk = y_chunk.var(axis=0)

        mean += mean_chunk * n
        var += var_chunk * n
        n_total += n

    mean /= n_total
    var /= n_total

    std = np.sqrt(var) + 1e-6

    return mean, std

In [ ]:
def get_Y_chunk(Y, rows, idx, mean, std):
    y = np.asarray(Y[rows[idx]], dtype=np.float32)
    return (y - mean) / std

In [ ]:
# load subject
info = info1  # from earlier

Y = np.load(info["Y_path"], mmap_mode="r")
rows = np.load(info["rows_path"])

# compute stats on TRAIN ONLY
mean, std = compute_Y_stats(Y, rows, train_idx)

# get normalized chunks
Y_train_chunk = get_Y_chunk(Y, rows, train_idx[:256], mean, std)
Y_test_chunk  = get_Y_chunk(Y, rows, test_idx[:256], mean, std)

print(Y_train_chunk.shape)

(256, 211339)


In [ ]:
print("X shape:", X.shape)
print("Final stimuli:", len(final_stimuli))

for sub in ["sub-01", "sub-02", "sub-03"]:
    rows = np.load(PROC / sub / f"Y_{sub}_rows.npy")
    print(sub, "rows:", len(rows))

X shape: (8003, 512)
Final stimuli: 8003
sub-01 rows: 8003
sub-02 rows: 8003
sub-03 rows: 8003


In [ ]:
for sub in ["sub-01", "sub-02", "sub-03"]:
    rows = np.load(PROC / sub / f"Y_{sub}_rows.npy")
    idx = pd.read_csv(PROC / sub / "image_index.csv")

    aligned = idx.iloc[rows]["stimulus"].values

    assert np.all(aligned == final_stimuli), f"{sub} misaligned!"

In [ ]:
print("X mean (train):", X_train.mean(), "std:", X_train.std())

X mean (train): 6.134667212763966e-18 std: 0.9999999999999998


In [ ]:
Y = np.load(info["Y_path"], mmap_mode="r")
rows = np.load(info["rows_path"])

y_train_sample = get_Y_chunk(Y, rows, train_idx[:1000], mean, std)

print("Y train sample mean:", y_train_sample.mean())
print("Y train sample std:", y_train_sample.std())

Y train sample mean: 0.0
Y train sample std: 0.0


In [ ]:
for sub in ["sub-01", "sub-02", "sub-03"]:
    print(f"\n===== {sub} =====")

    info = align_subject_Y_index(sub)

    Y = np.load(info["Y_path"], mmap_mode="r")
    rows = np.load(info["rows_path"])

    mean, std = compute_Y_stats(Y, rows, train_idx)

    y_train = get_Y_chunk(Y, rows, train_idx[:1000], mean, std)

    print("Y shape:", Y.shape)
    print("Mean:", y_train.mean())
    print("Std:", y_train.std())


===== sub-01 =====
sub-01: using 8003 aligned stimuli
sub-01 using DRIVE file
sub-01: index mapping ready → shape (8003,)
sub-01: saved row indices → /content/drive/MyDrive/things_fmri_project/processed_trials/sub-01/Y_sub-01_rows.npy
Y shape: (8740, 211339)
Mean: 0.0
Std: 0.0

===== sub-02 =====
sub-02: using 8003 aligned stimuli
sub-02 using TMP file
sub-02: index mapping ready → shape (8003,)
sub-02: saved row indices → /content/drive/MyDrive/things_fmri_project/processed_trials/sub-02/Y_sub-02_rows.npy
Y shape: (8740, 226950)
Mean: -0.0017510633
Std: 1.0033497

===== sub-03 =====
sub-03: using 8003 aligned stimuli
sub-03 using TMP file
sub-03: index mapping ready → shape (8003,)
sub-03: saved row indices → /content/drive/MyDrive/things_fmri_project/processed_trials/sub-03/Y_sub-03_rows.npy
Y shape: (8740, 189164)
Mean: -0.0033189624
Std: 1.0007666


In [ ]:
Y_path = TMP / "sub-01" / "Y_sub-01_images.npy"
Y = np.load(Y_path, mmap_mode="r")

rows = np.load(PROC / "sub-01" / "Y_sub-01_rows.npy")

print(Y.shape)
print(Y[rows[:10]].mean(), Y[rows[:10]].std())

FileNotFoundError: [Errno 2] No such file or directory: '/content/image_level_tmp/sub-01/Y_sub-01_images.npy'